# Baseline Comparison Study
## RealMLP, TabPFN, and Standard Stacking

This notebook implements the modern baseline comparisons.
All baselines use the pipeline (no data leakage).

## 1. Setup & Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import warnings
from datetime import datetime

from sklearn.model_selection import train_test_split
from sklearn.ensemble import (
    RandomForestRegressor, GradientBoostingRegressor,
    ExtraTreesRegressor, StackingRegressor
)
from sklearn.linear_model import Ridge
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler, PowerTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

# Configuration
RANDOM_STATE = 42
TEST_SIZE = 0.2
RESULTS_DIR = './results'
os.makedirs(RESULTS_DIR, exist_ok=True)

print(f"Baseline comparison started at: {datetime.now()}")

## 2. Data Loading

In [ ]:
# Load dataset
data_path = '../data/merged-data-all-continent.xlsx'
df = pd.read_excel(data_path)
df.columns = [str(x).strip() for x in df.columns]

# Define target parameters
target_params = [
    "pH_1-1_Water", "CEC(meq/100g)", "EC  μS/cm", "T.Nitrogen %",
    "T.Carbon %", "LOI %", "Clay %", "Sand %", "Silt %",
    "Calcium, ppm", "Copper, ppm", "Magnesium, ppm",
    "Phosphorus, ppm", "Potassium, ppm", "Sodium, ppm",
    "Sulfur, ppm", "Zinc, ppm"
]

# Define feature columns
pxrf_columns = ['K', 'Ca', 'Ti', 'V', 'Cr', 'Mn', 'Fe', 'Ni', 'Cu', 'Zn', 'As', 'Rb', 'Sr', 'Zr', 'Ba', 'Pb']
spectral_columns = [col for col in df.columns if col.isnumeric()]
feature_columns = spectral_columns + pxrf_columns

print(f"Dataset shape: {df.shape}")
print(f"Number of target parameters: {len(target_params)}")
print(f"Number of features: {len(feature_columns)}")

## 3. Data Preparation Function

In [ ]:
def prepare_data(df, target_param, feature_columns):
    """Prepare clean data for a target parameter."""
    target_df = df[df[target_param].notnull()].copy()
    target_df = target_df[[target_param] + feature_columns]

    for col in feature_columns:
        target_df[col] = pd.to_numeric(target_df[col], errors='coerce')

    target_df = target_df.dropna()

    X = target_df[feature_columns].values
    y = target_df[target_param].values

    return X, y


def create_standard_pipeline(model):
    pipeline = Pipeline([
        ('scaler', StandardScaler()),
        ('transformer', PowerTransformer(method='yeo-johnson', standardize=False)),
        ('model', model)
    ])
    return pipeline


def create_stacking_regressor(random_state=42):
    estimators = [
        ('rf', RandomForestRegressor(n_estimators=100, random_state=random_state)),
        ('gbr', GradientBoostingRegressor(n_estimators=100, random_state=random_state)),
        ('etr', ExtraTreesRegressor(n_estimators=100, random_state=random_state)),
    ]
    
    stacking = StackingRegressor(
        estimators=estimators,
        final_estimator=Ridge(alpha=1.0),
        cv=5,
        n_jobs=-1
    )
    
    return Pipeline([
        ('scaler', StandardScaler()),
        ('transformer', PowerTransformer(method='yeo-johnson', standardize=False)),
        ('model', stacking)
    ])


def create_realmlp_baseline(random_state=42):    
    from importlib import import_module
    pytabkit = import_module("pytabkit")
    model = pytabkit.RealMLP_TD_Regressor(random_state=random_state)
    print("  Using PyTabKit RealMLP_TD_Regressor")
    return model
    


def create_tabpfn_baseline():
    """
    Create a TabPFN baseline.
    """
    try:
        from tabpfn import TabPFNRegressor
        model = TabPFNRegressor()
        print("  Using TabPFN foundation model")
        return Pipeline([
            ('scaler', StandardScaler()),
            ('model', model)
        ])
    except ImportError:
        print("  WARNING: tabpfn package not installed. Skipping TabPFN baseline.")
        print("  Install with: pip install tabpfn")
        return None

## 4. Baseline Models Definition

In [ ]:
def prepare_data(df, target_param, feature_columns):
    """Prepare clean data for a target parameter."""
    target_df = df[df[target_param].notnull()].copy()
    target_df = target_df[[target_param] + feature_columns]

    for col in feature_columns:
        target_df[col] = pd.to_numeric(target_df[col], errors='coerce')

    target_df = target_df.dropna()

    X = target_df[feature_columns].values
    y = target_df[target_param].values

    return X, y


def create_standard_pipeline(model, random_state=42):
    pipeline = Pipeline([
        ('scaler', StandardScaler()),
        ('transformer', PowerTransformer(method='yeo-johnson', standardize=False)),
        ('model', model)
    ])
    return pipeline


def create_stacking_regressor(random_state=42):
    estimators = [
        ('rf', RandomForestRegressor(n_estimators=100, random_state=random_state)),
        ('gbr', GradientBoostingRegressor(n_estimators=100, random_state=random_state)),
        ('etr', ExtraTreesRegressor(n_estimators=100, random_state=random_state)),
    ]
    
    stacking = StackingRegressor(
        estimators=estimators,
        final_estimator=Ridge(alpha=1.0),
        cv=5,
        n_jobs=-1
    )
    
    return Pipeline([
        ('scaler', StandardScaler()),
        ('transformer', PowerTransformer(method='yeo-johnson', standardize=False)),
        ('model', stacking)
    ])


def create_realmlp_baseline(random_state=42):
    
    from importlib import import_module
    pytabkit = import_module("pytabkit")
    model = pytabkit.RealMLP_TD_Regressor(random_state=random_state)
    print("  Using PyTabKit RealMLP_TD_Regressor")
    return model
    


def create_tabpfn_baseline():
    """
    Create a TabPFN baseline.
    """
    try:
        from tabpfn import TabPFNRegressor
        model = TabPFNRegressor()
        print("  Using TabPFN foundation model")
        return Pipeline([
            ('scaler', StandardScaler()),
            ('model', model)
        ])
    except ImportError:
        print("  WARNING: tabpfn package not installed. Skipping TabPFN baseline.")
        print("  Install with: pip install tabpfn")
        return None

## 5. Run Baseline Comparisons for All Properties

In [ ]:
def evaluate_model(model, X_train, X_test, y_train, y_test, model_name):
    """Train and evaluate a model, return metrics."""
    try:
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        
        r2 = r2_score(y_test, y_pred)
        rmse = np.sqrt(mean_squared_error(y_test, y_pred))
        mae = mean_absolute_error(y_test, y_pred)
        bias = np.mean(y_pred - y_test)
        
        return {
            'Model': model_name,
            'R2': r2,
            'RMSE': rmse,
            'MAE': mae,
            'Bias': bias
        }
    except Exception as e:
        print(f"    Error with {model_name}: {e}")
        return {
            'Model': model_name,
            'R2': np.nan,
            'RMSE': np.nan,
            'MAE': np.nan,
            'Bias': np.nan
        }

In [ ]:
all_baseline_results = {}

for param in target_params:
    print(f"\n{'='*70}")
    print(f"Processing: {param}")
    print(f"{'='*70}")
    
    # Prepare data
    X, y = prepare_data(df, param, feature_columns)
    
    if len(X) < 50:
        print(f"  Insufficient data ({len(X)} samples). Skipping.")
        continue
    
    # CORRECTED: Split FIRST, before any preprocessing
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE
    )
    print(f"  Train: {len(X_train)}, Test: {len(X_test)}")
    
    param_results = []
    
    # Baseline 1: Random Forest (original baseline)
    print("  Training Random Forest...")
    rf_pipeline = create_standard_pipeline(
        RandomForestRegressor(n_estimators=100, random_state=RANDOM_STATE)
    )
    param_results.append(
        evaluate_model(rf_pipeline, X_train, X_test, y_train, y_test, 'Random Forest')
    )
    
    # Baseline 2: Gradient Boosting
    print("  Training Gradient Boosting...")
    gbr_pipeline = create_standard_pipeline(
        GradientBoostingRegressor(n_estimators=100, random_state=RANDOM_STATE)
    )
    param_results.append(
        evaluate_model(gbr_pipeline, X_train, X_test, y_train, y_test, 'Gradient Boosting')
    )
    
    # Baseline 3: Extra Trees
    print("  Training Extra Trees...")
    etr_pipeline = create_standard_pipeline(
        ExtraTreesRegressor(n_estimators=100, random_state=RANDOM_STATE)
    )
    param_results.append(
        evaluate_model(etr_pipeline, X_train, X_test, y_train, y_test, 'Extra Trees')
    )
    
    # Baseline 4: Standard Stacking (without learned gating)
    print("  Training Standard Stacking...")
    stacking_pipeline = create_stacking_regressor(random_state=RANDOM_STATE)
    param_results.append(
        evaluate_model(stacking_pipeline, X_train, X_test, y_train, y_test, 'Standard Stacking')
    )
    
    # Baseline 5: RealMLP (or approximation)
    print("  Training RealMLP baseline...")
    realmlp_pipeline = create_realmlp_baseline(random_state=RANDOM_STATE)
    param_results.append(
        evaluate_model(realmlp_pipeline, X_train, X_test, y_train, y_test, 'RealMLP')
    )
    
    # Baseline 6: TabPFN (if available)
    print("  Training TabPFN baseline...")
    tabpfn_pipeline = create_tabpfn_baseline()
    if tabpfn_pipeline is not None:
        # TabPFN has a limit on number of features/samples
        if X_train.shape[1] <= 100 and X_train.shape[0] <= 10000:
            param_results.append(
                evaluate_model(tabpfn_pipeline, X_train, X_test, y_train, y_test, 'TabPFN')
            )
        else:
            print(f"    TabPFN skipped: too many features ({X_train.shape[1]}) or samples ({X_train.shape[0]})")
            param_results.append({
                'Model': 'TabPFN', 'R2': np.nan, 'RMSE': np.nan,
                'MAE': np.nan, 'Bias': np.nan
            })
    
    # Print results for this parameter
    param_df = pd.DataFrame(param_results)
    print(f"\n  Results for {param}:")
    print(param_df.to_string(index=False))
    
    all_baseline_results[param] = param_df

## 6. Comprehensive Results Table

In [ ]:
# Build comprehensive comparison table
summary_rows = []

for param, param_df in all_baseline_results.items():
    for _, row in param_df.iterrows():
        summary_rows.append({
            'Parameter': param,
            'Model': row['Model'],
            'R2': row['R2'],
            'RMSE': row['RMSE'],
            'MAE': row['MAE'],
            'Bias': row['Bias']
        })

summary_df = pd.DataFrame(summary_rows)

print("\n" + "="*80)
print("COMPREHENSIVE BASELINE COMPARISON")
print("="*80)

# Pivot table: Parameters as rows, Models as columns, R² as values
if len(summary_df) > 0:
    pivot_r2 = summary_df.pivot_table(index='Parameter', columns='Model', values='R2')
    print("\nR² Comparison (Test Set):")
    print(pivot_r2.to_string())
    
    pivot_rmse = summary_df.pivot_table(index='Parameter', columns='Model', values='RMSE')
    print("\n\nRMSE Comparison (Test Set):")
    print(pivot_rmse.to_string())

# Save to Excel
baseline_path = os.path.join(RESULTS_DIR, 'revised_baseline_comparison.xlsx')
summary_df.to_excel(baseline_path, index=False)
print(f"\nResults saved to: {baseline_path}")

# Also save the pivot tables
with pd.ExcelWriter(os.path.join(RESULTS_DIR, 'revised_baseline_pivots.xlsx')) as writer:
    if len(summary_df) > 0:
        pivot_r2.to_excel(writer, sheet_name='R2_comparison')
        pivot_rmse.to_excel(writer, sheet_name='RMSE_comparison')

## 7. Visualization

In [ ]:
if len(summary_df) > 0:
    # Figure 1: R² comparison across all methods
    fig, ax = plt.subplots(figsize=(18, 8))
    
    models = summary_df['Model'].unique()
    params = summary_df['Parameter'].unique()
    x = np.arange(len(params))
    width = 0.8 / len(models)
    
    colors = plt.cm.Set2(np.linspace(0, 1, len(models)))
    
    for i, model in enumerate(models):
        model_data = summary_df[summary_df['Model'] == model]
        r2_values = []
        for param in params:
            val = model_data[model_data['Parameter'] == param]['R2'].values
            r2_values.append(val[0] if len(val) > 0 else np.nan)
        
        offset = (i - len(models)/2 + 0.5) * width
        ax.bar(x + offset, r2_values, width, label=model, alpha=0.85, color=colors[i])
    
    ax.set_xlabel('Soil Property', fontsize=12)
    ax.set_ylabel('R² (Test Set)', fontsize=12)
    ax.set_title('Baseline Comparison: R² Across All Soil Properties', fontsize=14)
    ax.set_xticks(x)
    ax.set_xticklabels(params, rotation=90, fontsize=8)
    ax.legend(loc='lower left', fontsize=9)
    ax.axhline(y=0.8, color='gray', linestyle='--', alpha=0.3)
    ax.set_ylim(bottom=-0.1)
    
    plt.tight_layout()
    plt.savefig(os.path.join(RESULTS_DIR, 'revised_baseline_r2_comparison.png'), dpi=150, bbox_inches='tight')
    plt.show()
    
    # Figure 2: R² vs. RMSE scatter for all methods
    fig, ax = plt.subplots(figsize=(12, 8))
    
    markers = ['o', 's', '^', 'D', 'v', 'P', '*']
    for i, model in enumerate(models):
        model_data = summary_df[summary_df['Model'] == model]
        ax.scatter(model_data['RMSE'], model_data['R2'],
                  label=model, marker=markers[i % len(markers)],
                  s=80, alpha=0.7, color=colors[i])
    
    ax.set_xlabel('RMSE (Test Set)', fontsize=12)
    ax.set_ylabel('R² (Test Set)', fontsize=12)
    ax.set_title('R² vs. RMSE: Baseline Comparison', fontsize=14)
    ax.legend(fontsize=9)
    
    plt.tight_layout()
    plt.savefig(os.path.join(RESULTS_DIR, 'revised_baseline_r2_vs_rmse.png'), dpi=150, bbox_inches='tight')
    plt.show()

print(f"\nBaseline comparison completed at: {datetime.now()}")
print("All results and figures saved.")